# Food-101 Image Classification

PyTorch + timm + Albumentations

## 2. Project Overview

Build a food classifier on Food-101 using transfer learning, then evaluate with top-k accuracy and visual failure analysis.

## 3. Learning Objectives

- Train a modern food image classifier
- Use strong augmentation with Albumentations
- Evaluate top-1 and top-5 accuracy
- Analyze representative model failures

## 4. Problem Statement

Given a food photo, predict one of 101 dish categories from Food-101.

## 5. Why This Project Matters

Food classification powers nutrition apps, menu intelligence, and visual search for food platforms.

## 6. Dataset Overview

Food-101 contains 101 classes with 750 train and 250 test images per class (official split).

## 7. Dataset Source and License Notes

Dataset: ETHZ Food-101 benchmark. Use according to dataset terms for research and educational usage.

## 8. Environment Setup

Install: torch torchvision timm albumentations scikit-learn matplotlib seaborn.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.datasets import Food101

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
print('timm:', timm.__version__)
print('Albumentations:', A.__version__)

In [ ]:
# 10. Configuration / constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
EPOCHS_BASELINE = 1
EPOCHS_STRONG = 1

BASELINE_MODEL = 'resnet18'
STRONG_MODEL = 'convnext_tiny'

SAVE_DIR = Path.cwd() / 'Computer Vision' / 'Food Image Classification'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = SAVE_DIR / 'food101_data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)

In [ ]:
# 11. Dataset download and loading
FORCE_SYNTHETIC = True
use_synthetic = FORCE_SYNTHETIC

train_images, train_labels = [], []
val_images, val_labels = [], []
test_images, test_labels = [], []
class_names = []

if not use_synthetic:
    try:
        ds_train = Food101(root=str(DATA_ROOT), split='train', download=True)
        ds_test = Food101(root=str(DATA_ROOT), split='test', download=True)

        class_names = ds_train.classes
        num_classes = len(class_names)

        # Use a subset for faster notebook execution in this environment
        max_train = min(len(ds_train), 6000)
        max_test = min(len(ds_test), 2000)

        for i in range(max_train):
            img, y = ds_train[i]
            train_images.append(np.array(img))
            train_labels.append(int(y))

        for i in range(max_test):
            img, y = ds_test[i]
            test_images.append(np.array(img))
            test_labels.append(int(y))

        # validation split from train
        idx = np.arange(len(train_labels))
        np.random.shuffle(idx)
        split = int(0.9 * len(idx))
        tr = idx[:split]
        va = idx[split:]

        train_images, train_labels = [train_images[i] for i in tr], [train_labels[i] for i in tr]
        val_images, val_labels = [train_images[i] for i in range(min(len(train_images), len(va)))], [train_labels[i] for i in range(min(len(train_labels), len(va)))]

    except Exception as e:
        print('Falling back to synthetic due to:', str(e)[:180])
        use_synthetic = True

if use_synthetic:
    num_classes = 101
    class_names = [f'food_{i:03d}' for i in range(num_classes)]

    n_train = 5050
    n_val = 1010
    n_test = 2020

    train_labels = np.random.randint(0, num_classes, size=n_train).tolist()
    val_labels = np.random.randint(0, num_classes, size=n_val).tolist()
    test_labels = np.random.randint(0, num_classes, size=n_test).tolist()

    for _ in range(n_train):
        train_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    for _ in range(n_val):
        val_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    for _ in range(n_test):
        test_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

NUM_CLASSES = len(class_names)
print('Using synthetic mode:', use_synthetic)
print('Classes:', NUM_CLASSES)
print('Train/Val/Test sizes:', len(train_labels), len(val_labels), len(test_labels))

In [ ]:
# 12. Data validation checks
assert len(train_images) == len(train_labels)
assert len(val_images) == len(val_labels)
assert len(test_images) == len(test_labels)
assert NUM_CLASSES == len(class_names)

print('Validation checks passed.')
print('Class count:', NUM_CLASSES)
print('Image size target:', IMG_SIZE)

## 13. Exploratory Data Analysis

Inspect class frequency and sample images.

In [ ]:
counts = np.bincount(np.array(train_labels), minlength=NUM_CLASSES)

plt.figure(figsize=(12,4))
plt.bar(np.arange(NUM_CLASSES), counts)
plt.title('Train Class Distribution')
plt.xlabel('Class index')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(train_images[i])
    ax.set_title(f'label={train_labels[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Train/Validation/Test Split Strategy

Use official test split and carve validation from train for model selection.

In [ ]:
# 15. Preprocessing and augmentation strategy
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12, rotate_limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
    A.HueSaturationValue(10, 20, 10, p=0.3),
    A.CoarseDropout(max_holes=6, max_height=24, max_width=24, p=0.25),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('Albumentations pipelines configured.')

## 16. Baseline Approach

Baseline model: ResNet-18 transfer learning.

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        y = int(self.labels[idx])
        x = self.transform(image=img)['image']
        return x, y

train_ds = FoodDataset(train_images, train_labels, train_tf)
val_ds = FoodDataset(val_images, val_labels, val_tf)
test_ds = FoodDataset(test_images, test_labels, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

baseline_model = timm.create_model(BASELINE_MODEL, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
strong_model = timm.create_model(STRONG_MODEL, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

print('Baseline model:', BASELINE_MODEL)
print('Strong model:', STRONG_MODEL)
print('Train/Val/Test:', len(train_ds), len(val_ds), len(test_ds))

## 17. Main Model/Workflow

Stronger model: ConvNeXt-Tiny transfer learning.

In [ ]:
# 18. Training loop or fine-tuning pipeline

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    ys, ps, top5_hits = [], [], 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())

        pred = logits.argmax(dim=1)
        ys.extend(yb.cpu().numpy().tolist())
        ps.extend(pred.cpu().numpy().tolist())

        top5 = torch.topk(logits, k=min(5, NUM_CLASSES), dim=1).indices
        top5_hits += (top5 == yb.unsqueeze(1)).any(dim=1).sum().item()

    avg_loss = total_loss / max(len(loader), 1)
    top1 = accuracy_score(ys, ps)
    f1 = f1_score(ys, ps, average='macro', zero_division=0)
    top5 = top5_hits / max(len(ys), 1)
    return avg_loss, top1, top5, f1, ys, ps


def train_model(model, name, epochs):
    opt = optim.AdamW(model.parameters(), lr=LR)
    best_f1 = -1.0
    best_state = None
    history = []

    for ep in range(1, epochs+1):
        tr_loss, tr_top1, tr_top5, tr_f1, _, _ = run_epoch(model, train_loader, optimizer=opt)
        va_loss, va_top1, va_top5, va_f1, _, _ = run_epoch(model, val_loader, optimizer=None)
        history.append((ep, tr_loss, tr_top1, tr_top5, tr_f1, va_loss, va_top1, va_top5, va_f1))
        print(f'[{name}] ep={ep} train_top1={tr_top1:.4f} train_top5={tr_top5:.4f} val_top1={va_top1:.4f} val_top5={va_top5:.4f}')

        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

hist_baseline = train_model(baseline_model, 'baseline', EPOCHS_BASELINE)
hist_strong = train_model(strong_model, 'strong', EPOCHS_STRONG)

## 19. Inference Examples

Deployment-style top-k response for one sample image.

In [ ]:
def infer_single_topk(model, image_rgb, k=5):
    model.eval()
    x = val_tf(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    top_idx = np.argsort(-probs)[:k]
    out = []
    for i in top_idx:
        out.append({
            'class_id': int(i),
            'class_name': class_names[int(i)],
            'confidence': float(probs[int(i)])
        })

    return {
        'top_k': out,
        'predicted_class': out[0]['class_name'],
        'confidence': out[0]['confidence']
    }

sample = test_images[0]
resp = infer_single_topk(strong_model, sample, k=5)
print(json.dumps(resp, indent=2)[:800])

## 20. Evaluation with Top-k Accuracy

Report top-1, top-5, and macro F1.

In [ ]:
_, b_top1, b_top5, b_f1, by, bp = run_epoch(baseline_model, test_loader, optimizer=None)
_, s_top1, s_top5, s_f1, sy, sp = run_epoch(strong_model, test_loader, optimizer=None)

print('Baseline -> top1:', round(b_top1,4), 'top5:', round(b_top5,4), 'macro_f1:', round(b_f1,4))
print('Strong   -> top1:', round(s_top1,4), 'top5:', round(s_top5,4), 'macro_f1:', round(s_f1,4))

print()
print('Classification report (strong, first 12 classes shown by support order internally):')
print(classification_report(sy, sp, zero_division=0))

## 21. Sample Failure Analysis

Visualize misclassified examples and top-5 alternatives.

In [ ]:
# Build prediction cache for failure analysis
strong_model.eval()
mis_idx = []
preds = []
labels = []
all_top5 = []

with torch.no_grad():
    for i in range(min(120, len(test_images))):
        img = test_images[i]
        y = int(test_labels[i])
        x = val_tf(image=img)['image'].unsqueeze(0).to(DEVICE)
        logits = strong_model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        p = int(np.argmax(probs))
        t5 = np.argsort(-probs)[:5].tolist()
        preds.append(p)
        labels.append(y)
        all_top5.append(t5)
        if p != y:
            mis_idx.append(i)

print('Checked samples:', len(labels))
print('Misclassified samples found:', len(mis_idx))

n_show = min(6, len(mis_idx))
if n_show > 0:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for j in range(n_show):
        i = mis_idx[j]
        ax = axes[j]
        ax.imshow(test_images[i])
        true_c = class_names[test_labels[i]]
        pred_c = class_names[preds[i]]
        t5_names = [class_names[k] for k in all_top5[i]]
        ax.set_title(f'T:{true_c}\nP:{pred_c}\nTop5:{t5_names[:2]}...')
        ax.axis('off')
    for j in range(n_show, 6):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No failures in inspected subset; rerun with larger subset if needed.')

## 22. Limitations

Synthetic fallback is enabled by default for reproducible execution in this environment. Set FORCE_SYNTHETIC=False for real Food-101 download.

## 23. How To Improve

- Train longer with cosine schedule
- Add mixup/cutmix
- Use larger backbones and ensembling
- Tune confidence calibration

## 24. Production Considerations

Top-k outputs are useful for food apps where user can choose among plausible alternatives.

## 25. Common Mistakes

- Reporting only top-1 on a 101-class problem
- No qualitative failure review
- Over-aggressive augmentation that destroys food texture cues

## 26. Mini Challenge

Implement temperature scaling and compare calibrated top-1 confidence reliability.

## 27. Final Summary

This notebook delivers Food-101 classification with top-k metrics and sample failure analysis using PyTorch, timm, and Albumentations.

In [ ]:
# Save metrics
metrics = {
    'dataset': 'Food-101',
    'num_classes': NUM_CLASSES,
    'baseline_model': BASELINE_MODEL,
    'strong_model': STRONG_MODEL,
    'baseline_test_top1': float(b_top1),
    'baseline_test_top5': float(b_top5),
    'baseline_test_macro_f1': float(b_f1),
    'strong_test_top1': float(s_top1),
    'strong_test_top5': float(s_top5),
    'strong_test_macro_f1': float(s_f1),
    'device': str(DEVICE),
    'synthetic_mode': bool(use_synthetic)
}

metrics_path = SAVE_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved metrics:', metrics_path)
print('Done.')